# Notebook 02: 方势垒模拟

## 高斯波包入射方势垒的含时演化

---

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
from config import SimParams, tunneling, above_barrier
from potentials import get_potential
from propagator import propagate
from visualization import (
    set_style, plot_potential_and_wavepacket,
    animate_evolution, plot_TR_evolution,
    plot_momentum_spectrum, plot_snapshots
)

set_style()

## 1. 隧穿区域 (E < V₀)

方势垒参数：$V_0 = 15$，$a = -1$，$b = 1$，势垒宽度 $L = 2$

波包参数：$k_0 = 5$，$E_{\text{kin}} = 12.5 < V_0$

预期：大部分反射，小部分隧穿透射

In [ ]:
p_tunnel = tunneling()
print(f'V0 = {p_tunnel.V0}, E_kinetic = {p_tunnel.E_kinetic:.2f}')
print(f'E/V0 = {p_tunnel.E_kinetic/p_tunnel.V0:.3f}')
print(f'Decay constant kappa = {np.sqrt(2*(p_tunnel.V0 - p_tunnel.E_kinetic)):.3f}')
print(f'Barrier width L = {p_tunnel.barrier_b - p_tunnel.barrier_a:.1f}')
print(f'Rough transmission T ~ exp(-2kappaL) approx {np.exp(-2*np.sqrt(2*(p_tunnel.V0-p_tunnel.E_kinetic))*(p_tunnel.barrier_b-p_tunnel.barrier_a)):.6f}')

In [ ]:
result_tunnel = propagate(p_tunnel, 'square')

In [ ]:
print(f'Final transmission: T = {result_tunnel.T_values[-1]:.6f}')
print(f'Final reflection: R = {result_tunnel.R_values[-1]:.6f}')
print(f'T + R = {result_tunnel.T_values[-1] + result_tunnel.R_values[-1]:.6f}')
print(f'Norm conservation: int|psi|^2dx = {result_tunnel.norm_values[-1]:.8f}')

### 波包演化快照

In [ ]:
fig = plot_snapshots(result_tunnel, n_snapshots=6, x_range=(-50, 50))
plt.show()

### 透射率/反射率演化

In [ ]:
fig = plot_TR_evolution(result_tunnel)
plt.show()

### 动量谱演化

In [ ]:
fig = plot_momentum_spectrum(result_tunnel, k_range=(-15, 15))
plt.show()

## 2. 经典透射区域 (E > V₀)

$V_0 = 5$，$E_{\text{kin}} = 12.5 > V_0$

预期：大部分透射，但仍存在量子反射

In [ ]:
p_above = above_barrier()
print(f'V0 = {p_above.V0}, E_kinetic = {p_above.E_kinetic:.2f}')
print(f'E/V0 = {p_above.E_kinetic/p_above.V0:.3f}')

In [ ]:
result_above = propagate(p_above, 'square')

In [ ]:
print(f'Final transmission: T = {result_above.T_values[-1]:.6f}')
print(f'Final reflection: R = {result_above.R_values[-1]:.6f}')
print(f'Quantum reflection fraction: {result_above.R_values[-1]/(result_above.T_values[-1]+result_above.R_values[-1])*100:.2f}%')

In [ ]:
fig = plot_snapshots(result_above, n_snapshots=6, x_range=(-50, 50))
plt.show()

In [ ]:
fig = plot_TR_evolution(result_above)
plt.show()

## 3. 势垒宽度对隧穿率的影响

In [ ]:
widths = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0]
T_widths = []

for w in widths:
    p_w = SimParams(V0=15.0, k0=5.0, barrier_a=-w/2, barrier_b=w/2)
    res = propagate(p_w, 'square')
    T_widths.append(res.T_values[-1])
    print(f'Width = {w:.1f}, T = {res.T_values[-1]:.6f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(widths, T_widths, 'bo-', linewidth=2, markersize=8, label='Numerical')

kappa = np.sqrt(2 * (15.0 - 0.5 * 5.0**2))
w_fine = np.linspace(0.3, 4.5, 100)
T_analytic = np.exp(-2 * kappa * w_fine)
ax.semilogy(w_fine, T_analytic, 'r--', linewidth=2, label=f'exp(-2kappaL), kappa={kappa:.2f}')

ax.set_xlabel('Barrier width L')
ax.set_ylabel('Transmission T')
ax.set_title('Tunneling rate vs Barrier width')
ax.legend()
plt.tight_layout()
plt.show()